In [1]:
import numpy as np
import pandas as pd
import tensorflow
import keras
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from scipy.sparse import triu
!pip install gensim
from gensim.models import Word2Vec
from scipy import spatial
import networkx as nx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 67.2 MB/s eta 0:00:00


In [2]:
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
text = """A sample text paragraph is a written section that illustrates a specific idea or theme. It typically includes a topic sentence, supporting details, and a concluding sentence. Sample paragraphs can be used for various purposes, such as demonstrating writing style, presenting information, or providing examples in educational contexts.

Structure of a Sample Text Paragraph
Topic Sentence: This is the first sentence that states the main idea of the paragraph. It sets the tone and direction for the rest of the text.
Supporting Sentences: These sentences provide evidence, examples, or explanations that bolster the main idea. They are essential for developing the paragraph’s theme and providing clarity.
Concluding Sentence: This wraps up the paragraph, reinforcing the main idea and providing a transition to the next point or paragraph.
Importance of Sample Text Paragraphs
Sample text paragraphs are vital for several reasons:

Clarity: Well-structured paragraphs help convey ideas clearly, making it easier for readers to understand the content.
Organization: They provide a framework for organizing thoughts, which is essential for effective communication.
Engagement: Engaging paragraphs capture the reader’s interest, encouraging them to continue reading."""

In [4]:
sentences = nltk.sent_tokenize(text)


clean_sentences = [re.sub(r'[^\w\s]',"",sentence.lower()) for sentence in sentences]


In [5]:
stopwords = stopwords.words("english")

clean_sentences = [[words for words in sentence.split()if words not in stopwords]for sentence in clean_sentences]


In [6]:
word_model = Word2Vec(clean_sentences, vector_size = 50, min_count=1, epochs = 100)

In [7]:
sentence_embedding = [[word_model.wv[word][0] for word in words]for words in clean_sentences]


In [8]:
max_length = max([len(sentence) for sentence in clean_sentences])

In [9]:
#Padding
sentence_embedding = [np.pad(embeddings,(0,max_length-len(embeddings)))  for embeddings in sentence_embedding]

In [10]:
similarity_matrix = np.zeros((len(sentences),len(sentences)))
for i, row in enumerate(sentence_embedding):
  for j, column in enumerate(sentence_embedding):
    similarity_matrix[i][j] = 1-spatial.distance.cosine(row,column)
    if similarity_matrix[i][j] < 0:
      similarity_matrix[i][j] = 0


In [11]:
nx_graph = nx.from_numpy_array(similarity_matrix)
scores = nx.pagerank(nx_graph,max_iter=1000)


In [12]:
top_sentence = {sentence:scores[index] for index, sentence in enumerate(sentences)}


top_four = dict(sorted(top_sentence.items(),key=lambda x:x[1], reverse=True)[:4])


In [13]:
for sentence in sentences:
  if sentence in top_four.keys():
    print(sentence)

A sample text paragraph is a written section that illustrates a specific idea or theme.
Sample paragraphs can be used for various purposes, such as demonstrating writing style, presenting information, or providing examples in educational contexts.
Supporting Sentences: These sentences provide evidence, examples, or explanations that bolster the main idea.
Organization: They provide a framework for organizing thoughts, which is essential for effective communication.
